In [ ]:
%%capture --no-stderr
!apt-get install tesseract-ocr-chi-sim
!pip install pytesseract pillow ftfy regex tqdm pandas openpyxl matplotlib
!pip install git+https://github.com/openai/CLIP.git
!apt-get install tesseract-ocr
print('Installing Chinese font package...')
!apt-get install -y fonts-wqy-zenhei
!tesseract --list-langs
!pip install opencv-python

In [ ]:
import os
import numpy as np
img_path = r"/content/drive/MyDrive/Lt-EDI_SJ/Train_images"
print(len(os.listdir(img_path)))
img_paths = [os.path.join(img_path, i) for i in os.listdir(img_path)]
img_paths.sort(key=lambda x: int(os.path.splitext(os.path.basename(x))[0]))
print(img_paths[35])

In [ ]:
# import numpy as np
# from tqdm import tqdm
# import pandas as pd

# # 1. Load the preprocessed test OCR results
# test_ocr_df = pd.read_csv('/content/drive/MyDrive/Lt-EDI_SJ/test_ocr_preprocessed.csv')

# # 2. Extract features using the defined extraction functions
# print("Extracting 1792-dim raw features for inference...")
# test_features_raw = []

# for index, row in tqdm(test_ocr_df.iterrows(), total=len(test_ocr_df)):
#     img_p = row['Image_Path']
#     txt = row['Processed_Text']

#     # Handle text NaNs
#     txt_str = str(txt) if pd.notna(txt) else ""

#     # Extract 1024-dim image features
#     img_f = extract_image_features(img_p)
#     # Extract 768-dim text features
#     txt_f = extract_text_features(txt_str)

#     # Concatenate to 1792-dim
#     combined_f = np.concatenate([img_f, txt_f])
#     test_features_raw.append(combined_f)

# X_inference_raw = np.array(test_features_raw)

# print(f"\nFeature extraction complete.")
# print(f"Inference feature shape: {X_inference_raw.shape}")

In [ ]:
import cv2
import os

# 1. Define input and output directories
test_input_dir = "/content/drive/MyDrive/Lt-EDI_SJ/chi_test"
test_output_dir = "/content/drive/MyDrive/Lt-EDI_SJ/Processed_test_images"

if not os.path.exists(test_output_dir):
    os.makedirs(test_output_dir)

# 2. Get list of test images
test_img_paths = [os.path.join(test_input_dir, f) for f in os.listdir(test_input_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))]

print(f"Starting image processing for {len(test_img_paths)} test images...")

# 3. Apply the specific processing logic
for i, img_path_single in enumerate(test_img_paths):
    img = cv2.imread(img_path_single)
    if img is None:
        print(f"Warning: Could not read image {img_path_single}. Skipping.")
        continue

    # Resize and process (alpha=1.2 for contrast, beta=20 for brightness)
    img_resized = cv2.resize(img, (224, 224))
    img_processed = cv2.convertScaleAbs(img_resized, alpha=1.2, beta=20)

    # Save processed image
    filename = os.path.basename(img_path_single)
    output_path = os.path.join(test_output_dir, filename)
    cv2.imwrite(output_path, img_processed)

    if (i + 1) % 50 == 0:
        print(f"Processed {i+1}/{len(test_img_paths)} images.")

print(f"All {len(test_img_paths)} images processed and saved to '{test_output_dir}'.")

In [ ]:
import pandas as pd
import re
import jieba
from sklearn.model_selection import train_test_split
# Predefined list of Chinese stopwords
chinese_stopwords = set([
    "的", "了", "在", "是", "我", "有", "和", "就", "不", "人", "都", "一", "一个", "上", "也", "很", "到", "说", "要", "去", "你", "会", "着", "没有", "看", "好", "自己", "这"
])


# Preprocess function for Chinese text
def preprocess_text(text, stopwords):
    # Remove URLs, special characters, etc.
    text = re.sub(r'http\S+', '', text)  # Remove URLs
    text = re.sub(r'[^\w\s]', '', text)  # Remove special characters
    text = re.sub(r'\d+', '', text)  # Remove numbers

    # Tokenize using jieba
    words = jieba.lcut(text)
    # Remove stopwords
    words = [word for word in words if word not in stopwords and len(word) > 1]
    return ' '.join(words)

In [ ]:
label_path = r"/content/drive/MyDrive/Lt-EDI_SJ/Train_labels.xlsx"
text_path = r"/content/drive/MyDrive/Lt-EDI_SJ/ocr_results_unprocessed.csv"
image_path = r'/content/drive/MyDrive/Lt-EDI_SJ/Processed_images'


# read and combine
df_labels = pd.read_excel(label_path)
df_text = pd.read_csv(text_path)
X = df_text['Extracted_Text']
y = df_labels['label'] # Corrected: 'Label' to 'label'

label_map = {
    "Homophobic": 0,
    "Transphobic": 1,
    "Non_Anti_LGBT": 2,
}
X = X.apply(lambda x: preprocess_text(str(x), chinese_stopwords)) # Corrected: apply directly to X, and convert to str to handle NaNs
y = y.map(label_map) # Corrected: directly apply map to the Series y

In [ ]:
from sklearn.model_selection import train_test_split

# Split the data into train and test sets (90% train, 10% test)
# We include img_paths, X (text), and y (labels) to keep them synchronized
img_train, img_test, X_train, X_test, y_train, y_test = train_test_split(
    img_paths, X, y, test_size=0.10, random_state=42, stratify=y
)

print(f"Training set size: {len(img_train)}")
print(f"Test set size: {len(img_test)}")

# Quick check of the first few labels to ensure consistency
print("\nSample training labels (first 5):")
print(y_train.head())


In [ ]:
!pip install transformers timm imbalanced-learn

In [ ]:
import torch
import torch.nn as nn
import timm
from transformers import AutoTokenizer, AutoModel
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from torchvision import transforms
from PIL import Image

print("Modules imported successfully.")

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. Initialize ConvNeXt model
print("Loading ConvNeXt model...")
img_model = timm.create_model('convnext_base', pretrained=True, num_classes=0).to(device)
img_model.eval()
for param in img_model.parameters():
    param.requires_grad = False

# 2. Define image transformation pipeline
img_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 3. Initialize ChineseBERT model and tokenizer
print("Loading ChineseBERT model...")
# Using 'bert-base-chinese' as a standard implementation of Chinese BERT
text_model_name = 'bert-base-chinese'
tokenizer = AutoTokenizer.from_pretrained(text_model_name)
text_model = AutoModel.from_pretrained(text_model_name).to(device)
text_model.eval()
for param in text_model.parameters():
    param.requires_grad = False

# 4. Function to extract image features
def extract_image_features(image_path):
    try:
        img = Image.open(image_path).convert('RGB')
        img_t = img_transforms(img).unsqueeze(0).to(device)
        with torch.no_grad():
            features = img_model(img_t)
        return features.cpu().numpy().flatten()
    except Exception as e:
        print(f"Error processing image {image_path}: {e}")
        return np.zeros(1024) # Return zero vector on error

# 5. Function to extract text features
def extract_text_features(text):
    inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = text_model(**inputs)
        # Use pooled output (CLS token representation)
        features = outputs.pooler_output
    return features.cpu().numpy().flatten()

# 6. Define linear projection layers
# convnext_base output dimension is 1024, bert-base-chinese is 768
img_feat_dim = 1024
text_feat_dim = 768
proj_dim = 512

img_projection = nn.Linear(img_feat_dim, proj_dim).to(device)
text_projection = nn.Linear(text_feat_dim, proj_dim).to(device)

print(f"Models loaded and projection layers initialized to dimension {proj_dim}.")

## Feature Extraction and Concatenation



In [ ]:
from tqdm import tqdm

def get_multimodal_features(img_paths, texts):
    combined_features = []
    for img_path, text in tqdm(zip(img_paths, texts), total=len(img_paths)):
        # 1. Extract raw features
        # Ensure text is a string
        text_str = str(text) if pd.notna(text) else ""

        img_feat = extract_image_features(img_path)
        text_feat = extract_text_features(text_str)

        # 2. Convert to tensors and move to device
        img_tensor = torch.tensor(img_feat, dtype=torch.float32).to(device).unsqueeze(0)
        text_tensor = torch.tensor(text_feat, dtype=torch.float32).to(device).unsqueeze(0)

        # 3. Apply projection layers
        with torch.no_grad():
            img_projected = img_projection(img_tensor)
            text_projected = text_projection(text_tensor)

            # 4. Concatenate features
            # result dim: (1, proj_dim + proj_dim)
            multimodal_feat = torch.cat((img_projected, text_projected), dim=1)

            # Store as numpy array
            combined_features.append(multimodal_feat.squeeze(0).cpu().numpy())

    return np.array(combined_features)

# Process training set
print("Extracting and projecting training features...")
X_train_multimodal = get_multimodal_features(img_train, X_train)

# Process test set
print("\nExtracting and projecting test features...")
X_test_multimodal = get_multimodal_features(img_test, X_test)

print(f"\nFeature extraction complete.")
print(f"Training set multimodal feature shape: {X_train_multimodal.shape}")
print(f"Test set multimodal feature shape: {X_test_multimodal.shape}")

In [ ]:
from collections import Counter

# 1. Print class distribution before SMOTE
print(f"Class distribution before SMOTE: {Counter(y_train)}")

# 2. Initialize SMOTE
smote = SMOTE(random_state=42)

# 3. Apply SMOTE to balance the training data
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_multimodal, y_train)

# 4. Print class distribution after SMOTE
print(f"Class distribution after SMOTE: {Counter(y_train_resampled)}")

print(f"\nShape of resampled training features: {X_train_resampled.shape}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. Instantiate Random Forest Classifier
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)

# 2. Train the model on the balanced training data
print("Training Random Forest classifier...")
rf_classifier.fit(X_train_resampled, y_train_resampled)

# 3. Make predictions on the test set
y_pred = rf_classifier.predict(X_test_multimodal)

# 4. Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"\nTest Accuracy: {accuracy:.4f}")

# 5. Generate and print classification report
target_names = ['Homophobic', 'Transphobic', 'Non_Anti_LGBT']
report = classification_report(y_test, y_pred, target_names=target_names)

print("\nClassification Report:")
print(report)

## Save Previous Model


In [ ]:
import joblib

# Define the save path
model_save_path = '/content/random_forest_baseline.joblib'

# Save the trained Random Forest classifier
joblib.dump(rf_classifier, model_save_path)

print(f"Model successfully saved to {model_save_path}")

Model successfully saved to /content/random_forest_baseline.joblib


In [ ]:
def get_raw_concatenated_features(img_paths, texts):
    raw_features = []
    for img_path, text in tqdm(zip(img_paths, texts), total=len(img_paths)):
        # Ensure text is a string
        text_str = str(text) if pd.notna(text) else ""

        # 1. Extract raw features (image: 1024, text: 768)
        img_feat = extract_image_features(img_path)
        text_feat = extract_text_features(text_str)

        # 2. Concatenate features into a 1792-dimensional vector
        # img_feat and text_feat are already numpy arrays from the extract functions
        concatenated_feat = np.concatenate([img_feat, text_feat])

        raw_features.append(concatenated_feat)

    return np.array(raw_features)

# Extract raw features for training set
print("Extracting raw features for training set...")
X_train_raw = get_raw_concatenated_features(img_train, X_train)

# Extract raw features for test set
print("\nExtracting raw features for test set...")
X_test_raw = get_raw_concatenated_features(img_test, X_test)

# Verify shapes
print(f"\nX_train_raw shape: {X_train_raw.shape}")
print(f"X_test_raw shape: {X_test_raw.shape}")

assert X_train_raw.shape[1] == 1792, f"Expected 1792 features, got {X_train_raw.shape[1]}"
assert X_test_raw.shape[1] == 1792, f"Expected 1792 features, got {X_test_raw.shape[1]}"
print("Feature extraction and verification successful.")

## SMOTE

In [ ]:
from collections import Counter
from imblearn.over_sampling import SMOTE

# 1. Verify current class distribution
print(f"Class distribution before SMOTE: {Counter(y_train)}")

# 2. Initialize SMOTE
smote_raw = SMOTE(random_state=42)

# 3. Apply SMOTE to raw training features
X_train_resampled_raw, y_train_resampled_raw = smote_raw.fit_resample(X_train_raw, y_train)

# 4. Confirm balanced class distribution
print(f"Class distribution after SMOTE: {Counter(y_train_resampled_raw)}")

# 5. Verify feature dimensionality
print(f"\nShape of resampled raw training features: {X_train_resampled_raw.shape}")
assert X_train_resampled_raw.shape[1] == 1792, f"Expected 1792 features, got {X_train_resampled_raw.shape[1]}"
print("Verification successful.")

In [ ]:
import torch
import torch.nn as nn

class MultimodalClassifier(nn.Module):
    def __init__(self, img_in_dim=1024, text_in_dim=768, proj_dim=512, num_classes=3):
        super(MultimodalClassifier, self).__init__()

        # 1. Image projection layer
        self.img_projection = nn.Linear(img_in_dim, proj_dim)

        # 2. Text projection layer
        self.text_projection = nn.Linear(text_in_dim, proj_dim)

        # 3. Activation function
        self.relu = nn.ReLU()

        # 4. Classification head
        # Concatenated dimension: proj_dim + proj_dim
        self.classifier = nn.Linear(proj_dim * 2, num_classes)

    def forward(self, x):
        # x shape: (batch_size, 1792)
        # Split input into image (1024) and text (768) features
        img_features = x[:, :1024]
        text_features = x[:, 1024:]

        # Project and apply activation
        img_projected = self.relu(self.img_projection(img_features))
        text_projected = self.relu(self.text_projection(text_features))

        # Concatenate projected features
        combined = torch.cat((img_projected, text_projected), dim=1)

        # Pass through classification head
        logits = self.classifier(combined)
        return logits

# Instantiate the model and move to device
model = MultimodalClassifier().to(device)
print("MultimodalClassifier model defined and instantiated on device:", device)
print(model)

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim
import copy

# 1. Prepare DataLoaders
# Convert numpy arrays/Series to Tensors
X_train_tensor = torch.tensor(X_train_resampled_raw, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_resampled_raw.values, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_raw, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 2. Define Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 3. Training Loop with Early Stopping
num_epochs = 100
patience = 10
best_val_loss = float('inf')
epochs_no_improve = 0
best_model_wts = copy.deepcopy(model.state_dict())

print("Starting training...")
for epoch in range(num_epochs):
    # Training phase
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)

    epoch_train_loss = running_loss / len(train_loader.dataset)

    # Validation (test set) phase
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)

    epoch_val_loss = val_loss / len(test_loader.dataset)

    # Early Stopping check
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_model_wts = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1}. Best Val Loss: {best_val_loss:.4f}")
            break

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {epoch_train_loss:.4f}, Val Loss: {epoch_val_loss:.4f}")

# Load best model weights
model.load_state_dict(best_model_wts)
print("Training complete.")

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

# 1. Evaluate the best model on the test set
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

# 2. Calculate accuracy
accuracy = accuracy_score(all_labels, all_preds)
print(f"Final Model Test Accuracy: {accuracy:.4f}")

# 3. Generate and print classification report
target_names = ['Homophobic', 'Transphobic', 'Non_Anti_LGBT']
report = classification_report(all_labels, all_preds, target_names=target_names)

print("\nClassification Report for Neural Network:")
print(report)

# save model
torch.save(model.state_dict(), '/content/multimodal_classifier_model1.pth')

#

In [ ]:
import torch
import torch.nn as nn

class MultimodalClassifier(nn.Module):
    def __init__(self, img_in_dim=1024, text_in_dim=768, proj_dim=256, num_classes=3):
        super(MultimodalClassifier, self).__init__()

        # 1. Linear projection layers
        self.img_projection = nn.Linear(img_in_dim, proj_dim)
        self.text_projection = nn.Linear(text_in_dim, proj_dim)

        # 2. Hidden layers
        self.hidden1 = nn.Linear(proj_dim * 2, 128) # 256 + 256 = 512
        self.hidden2 = nn.Linear(128, 32)

        # 3. Output layer
        self.output = nn.Linear(32, num_classes)

        # Activation function
        self.relu = nn.ReLU()

    def forward(self, x):
        # x shape: (batch_size, 1792)
        # 4. Split incoming tensor into image (1024) and text (768) components
        img_features = x[:, :1024]
        text_features = x[:, 1024:]

        # Project and apply activation
        img_projected = self.relu(self.img_projection(img_features))
        text_projected = self.relu(self.text_projection(text_features))

        # Concatenate projected features (Result dim: 512)
        combined = torch.cat((img_projected, text_projected), dim=1)

        # Pass through hidden layers with ReLU
        x = self.relu(self.hidden1(combined))
        x = self.relu(self.hidden2(x))

        # Final output layer
        logits = self.output(x)
        return logits

# 5. Instantiate the model and move to device
model = MultimodalClassifier().to(device)
print("Redefined MultimodalClassifier instantiated on device:", device)
print(model)

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim
import copy
from sklearn.metrics import classification_report, accuracy_score

# 1. Prepare DataLoaders
X_train_tensor = torch.tensor(X_train_resampled_raw, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_resampled_raw.values, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_raw, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# 2. Define Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 3. Training Loop with Strict Early Stopping
num_epochs = 100
best_val_loss = float('inf')
best_model_wts = copy.deepcopy(model.state_dict())

print("Starting training with strict early stopping...")
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)

    epoch_train_loss = running_loss / len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)

    epoch_val_loss = val_loss / len(test_loader.dataset)

    print(f"Epoch {epoch+1}: Train Loss: {epoch_train_loss:.4f}, Val Loss: {epoch_val_loss:.4f}")

    # Strict Early Stopping: Stop if validation loss increases
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_model_wts = copy.deepcopy(model.state_dict())
    else:
        print(f"Validation loss increased from {best_val_loss:.4f} to {epoch_val_loss:.4f}. Stopping training.")
        break

# Load best model weights
model.load_state_dict(best_model_wts)

# 4. Evaluation
model.eval()
all_preds = []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())

accuracy = accuracy_score(y_test, all_preds)
print(f"\nFinal Test Accuracy: {accuracy:.4f}")
print("Classification Report:")
print(classification_report(y_test, all_preds, target_names=['Homophobic', 'Transphobic', 'Non_Anti_LGBT']))

In [ ]:
import copy
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim

# 1. Re-instantiate the model
model = MultimodalClassifier().to(device)
print("MultimodalClassifier re-instantiated.")

# 2. Prepare DataLoaders
X_train_tensor = torch.tensor(X_train_resampled_raw, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_resampled_raw.values, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_raw, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 3. Define Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 4. Training Loop with Early Stopping (Patience = 5)
num_epochs = 100
patience = 5
best_val_loss = float('inf')
epochs_no_improve = 0
best_model_wts = copy.deepcopy(model.state_dict())

print("Starting training with patience 5...")
for epoch in range(num_epochs):
    # Training
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
    epoch_train_loss = running_loss / len(train_loader.dataset)

    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
    epoch_val_loss = val_loss / len(test_loader.dataset)

    # Early Stopping Logic
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_model_wts = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {epoch_train_loss:.4f}, Val Loss: {epoch_val_loss:.4f}")

    if epochs_no_improve >= patience:
        print(f"Early stopping triggered at epoch {epoch+1}. Best Val Loss: {best_val_loss:.4f}")
        break

# Load best model weights
model.load_state_dict(best_model_wts)
print("Training complete and best weights loaded.")

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

# 1. Evaluate the model on the test set
model.eval()
all_preds = []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())

# 2. Calculate accuracy
accuracy = accuracy_score(y_test, all_preds)
print(f"Final Test Accuracy (Patience 5): {accuracy:.4f}")

# 3. Generate and print classification report
target_names = ['Homophobic', 'Transphobic', 'Non_Anti_LGBT']
report = classification_report(y_test, all_preds, target_names=target_names)

print("\nClassification Report for Neural Network (Patience 5):")
print(report)

# 4. Save the model
torch.save(model.state_dict(), '/content/multimodal_classifier_patience5.pth')
print("Model saved to /content/multimodal_classifier_patience5.pth")

# Test image Result

In [ ]:
test_image_path ="/content/drive/MyDrive/Lt-EDI_SJ/chi_test"

In [ ]:
import pytesseract
from PIL import Image
import os
import pandas as pd
from tqdm import tqdm

# 1. List all images in the test directory
test_images = [os.path.join(test_image_path, f) for f in os.listdir(test_image_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))]
test_images.sort()

results = []

print(f'Starting OCR for {len(test_images)} images...')

# 2. Perform OCR and Preprocessing
for img_p in tqdm(test_images):
    try:
        # Open image
        img = Image.open(img_p)
        # Extract text (using simplified Chinese model)
        raw_text = pytesseract.image_to_string(img, lang='chi_sim')

        # Preprocess text using the function defined in cell aAoBzH4y1um-
        # Note: preprocess_text and chinese_stopwords are already in the global state
        clean_text = preprocess_text(str(raw_text), chinese_stopwords)

        results.append({
            'Image_Path': img_p,
            'Raw_Text': raw_text.strip(),
            'Processed_Text': clean_text
        })
    except Exception as e:
        print(f'Error processing {img_p}: {e}')
        results.append({
            'Image_Path': img_p,
            'Raw_Text': '',
            'Processed_Text': ''
        })

# 3. Create DataFrame and Save
df_test_results = pd.DataFrame(results)
save_csv_path = '/content/drive/MyDrive/Lt-EDI_SJ/test_ocr_preprocessed.csv'
df_test_results.to_csv(save_csv_path, index=False)

print(f'\nProcessing complete. Results saved to: {save_csv_path}')
display(df_test_results.head())

In [ ]:
test_processed_image = "/content/drive/MyDrive/Lt-EDI_SJ/Processed_test_images"
test_ocr_text = "/content/drive/MyDrive/Lt-EDI_SJ/test_ocr_preprocessed.csv"

In [ ]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

# 1. Load the preprocessed test OCR results
test_ocr_df = pd.read_csv('/content/drive/MyDrive/Lt-EDI_SJ/test_ocr_preprocessed.csv')

# 2. Create a list of full paths for the processed test images
# We assume the order in the CSV matches the images in the processed directory
# and that the CSV 'Image_Path' filename matches the processed image filenames
test_processed_dir = '/content/drive/MyDrive/Lt-EDI_SJ/Processed_test_images'
test_img_paths = [os.path.join(test_processed_dir, os.path.basename(p)) for p in test_ocr_df['Image_Path']]

# 3 & 4 & 5. Extract features and concatenate
print('Extracting 1792-dim raw features for inference...')
test_features_raw = []

for i, row in tqdm(test_ocr_df.iterrows(), total=len(test_ocr_df)):
    img_p = test_img_paths[i]
    txt = row['Processed_Text']

    # Handle text NaNs
    txt_str = str(txt) if pd.notna(txt) else ""

    # Extract 1024-dim image features (using frozen ConvNeXt)
    img_f = extract_image_features(img_p)
    # Extract 768-dim text features (using frozen ChineseBERT)
    txt_f = extract_text_features(txt_str)

    # Concatenate to 1792-dim
    combined_f = np.concatenate([img_f, txt_f])
    test_features_raw.append(combined_f)

X_inference_raw = np.array(test_features_raw)

# 6. Print final shape
print(f'\nFeature extraction complete.')
print(f'Inference feature shape: {X_inference_raw.shape}')
assert X_inference_raw.shape[1] == 1792, f'Expected 1792 features, got {X_inference_raw.shape[1]}'

In [ ]:
import torch
import pandas as pd
from collections import Counter

# 1. Re-instantiate the model with the same architecture
model_inf = MultimodalClassifier(img_in_dim=1024, text_in_dim=768, proj_dim=256, num_classes=3).to(device)

# 2. Load the trained weights
weights_path = '/content/multimodal_classifier_patience5.pth'
model_inf.load_state_dict(torch.load(weights_path, weights_only=True))
model_inf.eval()
print(f"Model weights loaded from {weights_path}.")

# 3. Prepare inference data
X_inf_tensor = torch.tensor(X_inference_raw, dtype=torch.float32).to(device)

# 4. Perform Inference
with torch.no_grad():
    logits = model_inf(X_inf_tensor)
    _, predicted_indices = torch.max(logits, 1)
    predicted_indices = predicted_indices.cpu().numpy()

# 5. Map indices to labels
# Original mapping: {'Homophobic': 0, 'Transphobic': 1, 'Non_Anti_LGBT': 2}
inv_label_map = {0: 'Homophobic', 1: 'Transphobic', 2: 'Non_Anti_LGBT'}
predicted_labels = [inv_label_map[idx] for idx in predicted_indices]

# 6. Create results DataFrame
test_ocr_df = pd.read_csv('/content/drive/MyDrive/Lt-EDI_SJ/test_ocr_preprocessed.csv')
inference_results = pd.DataFrame({
    'Image_Name': test_ocr_df['Image_Path'].apply(lambda x: os.path.basename(x)),
    'Predicted_Label': predicted_labels
})

# 7. Display Results
print("\nPrediction Distribution:")
print(Counter(predicted_labels))

print("\nFirst 10 Predictions:")
display(inference_results.head(10))

In [ ]:
import re

# 1. Prepare the final submission dataframe
# Extract numeric id from Image_Name (e.g., '10.jpg' -> 10)
submission_df = inference_results.copy()
submission_df['id'] = submission_df['Image_Name'].apply(lambda x: int(re.search(r'\d+', x).group()))

# 2. Rename columns as requested
submission_df = submission_df.rename(columns={'Predicted_Label': 'label'})

# 3. Select and reorder columns
submission_df = submission_df[['id', 'label']]

# 4. Sort by ID to ensure order from 1 to 239 (or 232 as per the test set size)
submission_df = submission_df.sort_values(by='id')

# 5. Save to CSV
output_csv_path = '/content/submission_results.csv'
submission_df.to_csv(output_csv_path, index=False)

print(f'Submission file saved to: {output_csv_path}')
display(submission_df.head())
print(f'Total rows: {len(submission_df)}')

In [ ]:
import shutil
import os

# Define source and destination paths
source_path = '/content/submission_results.csv'
destination_dir = '/content/drive/MyDrive/Lt-EDI_SJ/'
destination_path = os.path.join(destination_dir, 'submission_results.csv')

# Ensure the destination directory exists (though it should based on previous steps)
if os.path.exists(destination_dir):
    shutil.copy(source_path, destination_path)
    print(f'File successfully copied to Drive: {destination_path}')
else:
    print(f'Error: Destination directory {destination_dir} does not exist.')